# Benchmark: Kafka partitions × Spark workers for the Wikimedia pipeline

**Big Data & Predictive Analytics (ST153) — Q2 (RDD) and Q3 (DataFrame/SQL) performance study**

This notebook measures the *same* Wikipedia edit-processing pipeline that Q2 and Q3 implement, but as a
controlled experiment: the snapshot is replayed into benchmark topics with **P = 1, 2, 4, 8 Kafka
partitions**, and each topic is processed with **W = 1, 2, 4 Spark executors** (2 cores each, capped with
`spark.cores.max`). For every cell of the matrix it records

| metric | how it is obtained |
|---|---|
| **stage latency** | wall clock per pipeline stage (Spark job group) and per-task duration percentiles from the driver's Spark REST API |
| **throughput** | 200,000 messages ÷ end-to-end makespan (msg/s) |
| **speedup** | `T(W=1, P) / T(W, P)` for the same partition count |
| **efficiency** | `speedup / workers_active` (and against the requested W) |
| **weight distribution** | messages per Kafka partition, tasks and task-runtime per executor, Jain fairness index |
| **resources** | driver CPU/RAM/network/disk (psutil), executor memory/GC/core-seconds (Spark REST), cluster utilisation |
| **Kafka microbenchmark** | produce→consume latency p50/p95/p99 per partition count and producer configuration, producer and consumer throughput |

The pipeline stages map to the exam questions:

| stage | what runs | question |
|---|---|---|
| S1 ingest | `spark.read.format("kafka")` batch read of the whole snapshot, materialised | Q2 opening / Q3a read |
| S2 parse | RDD `json.loads` → 10-tuple, failures counted with an accumulator, `filter`, `count` | Q2a |
| S3 `reduceByKey` | top-10 wikis by human edits | Q2b-i, Q2c |
| S4 `groupByKey` | the same result through `groupByKey().mapValues(sum)` | Q2c |
| S5 SQL aggregate | explicit schema, `selisih_byte`, bot/human share per wiki | Q3a, Q3b-2 |
| S6 SQL window | 5-minute `window()` page-edit counts (spike detection) | Q3c |

The corpus is a controlled replay of the real `wiki.recentchange.raw` snapshot (109,868 events captured by
the Q1 producer), cycled to **200,000** messages so the Q2 size requirement is met; the copies get a bumped
`id`/`timestamp` so no two records are identical. Partition assignment for the balanced topics is explicit
round-robin, so the only variable between topics is **P**.

## Experimental design

**Controlled variables** — the same 200,000 payloads in every topic, the same code path for every cell, the
same `spark.sql.shuffle.partitions=16`, AQE on, 2 cores per executor, 1200 MB per executor.

**Independent variables**

* **P ∈ {1, 2, 4, 8}** — number of Kafka partitions (one topic created per P, `retention.ms=1h`).
* **W ∈ {1, 2, 4}** — number of Spark executors, enforced with `spark.cores.max = 2·W`. A Spark context
  cannot change its core budget after creation, so each W is a separate application; the first job of
  every application is a discarded warm-up (executors fetch the connector jars on first use).

**Why the matrix matters.** A Kafka partition is the unit of parallelism on the ingest side, a Spark core
is the unit on the compute side. If P < W some executors have nothing to read and the pipeline is
*ingest-starved*; if P ≫ W each executor serially handles several partitions and the shuffle dominates.
The matrix exposes both regimes, and the `reduceByKey` / `groupByKey` pair shows how much of the cost is
shuffle rather than scan.

**Derived quantities**

* throughput `X = N / T` (msg/s), speedup `S(W) = T(1,P) / T(W,P)`
* efficiency `E = S / W_active`, Jain fairness `J = (Σxᵢ)² / (n · Σxᵢ²)` over per-executor task counts
* cluster utilisation `U = Σ task CPU-runtime / (cores · makespan)`
* Little's law sanity check `L = X · latency` (tasks in flight at the mean)

In [1]:
# ============================== 1. Configuration, helpers, instrumentation ==============================
import base64, contextlib, json, os, re, socket, statistics, threading, time, uuid, urllib.request
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import psutil
from confluent_kafka import Consumer, Producer, TopicPartition
from confluent_kafka.admin import AdminClient, NewTopic

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 190)
pd.set_option("display.max_columns", 60)

BOOTSTRAP   = "10.8.0.6:29092"          # broker on the master PC (external listener)
SOURCE      = "wiki.recentchange.raw"   # snapshot produced for Q1
MASTER_UI   = "http://10.8.0.6:8080"
PREFIX      = "wiki.bench"
CORPUS_N    = 200_000
P_LIST      = [1, 2, 4, 8]
W_LIST      = [1, 2, 4]
BENCH_DIR   = Path("/tmp/wiki-bench")
PLOT_DIR    = BENCH_DIR / "plots"
RESULT_JSON = BENCH_DIR / "bench_results.json"
MICRO_JSON  = BENCH_DIR / "kafka_micro.json"
REPORT_HTML = BENCH_DIR / "wiki_q2_q3_benchmark.html"
TOPIC_COUNTS = {}                       # topic -> messages per partition
for _p in (BENCH_DIR, PLOT_DIR):
    _p.mkdir(parents=True, exist_ok=True)

STARTED = datetime.now()
print("notebook start:", STARTED.strftime("%Y-%m-%d %H:%M:%S"), "| host:", socket.gethostname())
print("python:", os.sys.version.split()[0], "| cpu:", psutil.cpu_count(),
      "| ram:", round(psutil.virtual_memory().total / 2**30, 1), "GiB")


# ------------------------------------------------------------------ REST helpers
def http_json(url, timeout=20):
    with urllib.request.urlopen(url, timeout=timeout) as fh:
        return json.load(fh)


def ui_json(sc, path, timeout=30):
    """Spark driver REST API - the same numbers the Spark UI shows."""
    base = sc.uiWebUrl.rstrip("/") + "/api/v1/applications/" + sc.applicationId
    return http_json(base + path, timeout)


STAGE_LOG = []


def measure(sc, label, fn):
    """Run one Spark action inside a job group and capture stage + task level metrics.

    Returns whatever `fn` returns. Appends one record to STAGE_LOG with wall clock,
    job/stage/task counts, shuffle bytes, per-task duration percentiles and the task
    distribution over executors (executorId -> tasks, executorId -> CPU ms).
    """
    tag = "bench::%s::%s" % (uuid.uuid4().hex[:8], label)
    with contextlib.suppress(TypeError):
        sc.setJobGroup(tag, label, False)
    with contextlib.suppress(Exception):
        sc.setJobGroup(tag, label)

    t0 = time.perf_counter()
    value = fn()
    wall = time.perf_counter() - t0

    try:
        job_ids = set(sc.statusTracker().getJobIdsForGroup(tag))
    except Exception:
        job_ids = set()

    jobs, stages = [], {}
    with contextlib.suppress(Exception):
        jobs = ui_json(sc, "/jobs")
        stages = {s["stageId"]: s for s in ui_json(sc, "/stages")}
    mine = [j for j in jobs if j["jobId"] in job_ids]
    sids = sorted({sid for j in mine for sid in j.get("stageIds", [])})

    durs, by_exec, ms_exec = [], Counter(), Counter()
    sw = sr = tasks = spilled = 0
    for sid in sids:
        st = stages.get(sid)
        if not st:
            continue
        tasks += st.get("numTasks", 0)
        sw += st.get("shuffleWriteBytes", 0) or 0
        sr += st.get("shuffleReadBytes", 0) or 0
        spilled += (st.get("memoryBytesSpilled", 0) or 0) + (st.get("diskBytesSpilled", 0) or 0)
        try:
            tl = ui_json(sc, "/stages/%d/%d/taskList" % (sid, st.get("attemptId", 0)))
        except Exception:
            tl = []
        for t in tl:
            durs.append(t.get("duration", 0))
            ex = str(t.get("executorId", t.get("host", "?")))
            by_exec[ex] += 1
            ms_exec[ex] += t.get("executorRunTime", t.get("duration", 0))

    def q(p):
        return round(float(np.percentile(durs, p)) / 1000.0, 3) if durs else None

    rec = {"label": label, "seconds": round(wall, 3), "jobs": len(mine), "stages": len(sids),
           "tasks": tasks, "shuffle_write": sw, "shuffle_read": sr, "spilled": spilled,
           "task_p50_ms": q(50), "task_p95_ms": q(95),
           "task_max_ms": round(max(durs) / 1000.0, 3) if durs else None,
           "by_executor": dict(by_exec), "cpu_ms_by_executor": dict(ms_exec),
           "at": datetime.now().strftime("%H:%M:%S")}
    STAGE_LOG.append(rec)
    return value


class ResourceSampler:
    """Samples the driver host (broker + Spark driver live on 10.8.0.6) while a run executes."""

    def __init__(self, interval=0.25):
        self.interval = interval
        self._stop = threading.Event()
        self._rows = []
        self._proc = psutil.Process()

    def _run(self):
        psutil.cpu_percent(interval=None)
        n0 = psutil.net_io_counters()
        d0 = psutil.disk_io_counters()
        while not self._stop.is_set():
            n = psutil.net_io_counters()
            d = psutil.disk_io_counters()
            self._rows.append({
                "cpu": psutil.cpu_percent(interval=None),
                "rss_mb": self._proc.memory_info().rss / 2**20,
                "sys_ram_pct": psutil.virtual_memory().percent,
                "net_rx_mb": (n.bytes_recv - n0.bytes_recv) / 2**20,
                "net_tx_mb": (n.bytes_sent - n0.bytes_sent) / 2**20,
                "disk_r_mb": (d.read_bytes - d0.read_bytes) / 2**20,
                "disk_w_mb": (d.write_bytes - d0.write_bytes) / 2**20,
            })
            time.sleep(self.interval)

    def start(self):
        self._th = threading.Thread(target=self._run, daemon=True)
        self._th.start()
        return self

    def stop(self):
        self._stop.set()
        self._th.join(timeout=5)
        rows = self._rows or [{}]
        return {
            "cpu_mean": round(statistics.mean(r.get("cpu", 0) for r in rows), 1),
            "cpu_max": round(max(r.get("cpu", 0) for r in rows), 1),
            "rss_mb": round(max(r.get("rss_mb", 0) for r in rows), 1),
            "sys_ram_pct": round(max(r.get("sys_ram_pct", 0) for r in rows), 1),
            "net_rx_mb": round(rows[-1].get("net_rx_mb", 0), 1),
            "net_tx_mb": round(rows[-1].get("net_tx_mb", 0), 1),
            "disk_r_mb": round(rows[-1].get("disk_r_mb", 0), 1),
            "disk_w_mb": round(rows[-1].get("disk_w_mb", 0), 1),
            "samples": len(rows),
        }


def jain(values):
    vals = [float(v) for v in values]
    s = sum(vals)
    return round((s * s) / (len(vals) * sum(v * v for v in vals)), 3) if s else 0.0


def table(df, title=""):
    if title:
        print(title)
    print(df.to_string(index=False))
    print()
    return df


print("helpers ready")

notebook start: 2026-11-17 10:37:57 | host: jupyter
python: 3.10.12 | cpu: 16 | ram: 23.1 GiB
helpers ready


## 1. Cluster and environment

The Spark standalone cluster is the one the Q2/Q3 notebooks already use: `spark://10.8.0.6:7077`, workers
registered with the master, 2 cores / 2048 MB each. The Jupyter kernel is the driver
(`spark.driver.host=10.8.0.6`, the broker PC). Executor capacity per application is capped with
`spark.cores.max`, which is how **W** is controlled.

In [2]:
# ============================== 2. Cluster roster ==============================
roster = http_json(MASTER_UI + "/json/")
workers = pd.DataFrame([{
    "host": w["host"], "cores": w["cores"], "memory_MB": w["memory"],
    "state": w["state"], "ui": w["webuiaddress"],
} for w in roster["workers"]])
table(workers, "workers registered with the master")
print("alive workers:", roster["aliveworkers"], "| total cores:", roster["cores"],
      "| total memory:", roster["memory"], "MB")

admin = AdminClient({"bootstrap.servers": BOOTSTRAP})
src_parts = sorted(admin.list_topics(timeout=10).topics[SOURCE].partitions)
print("\nsource topic %s: %d partitions" % (SOURCE, len(src_parts)))

c_probe = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": "bench-probe-" + uuid.uuid4().hex[:6],
                    "enable.auto.commit": False})
src_counts = {}
for p in src_parts:
    lo, hi = c_probe.get_watermark_offsets(TopicPartition(SOURCE, p), timeout=5, cached=False)
    src_counts[p] = hi - lo
c_probe.close()
print("source snapshot messages:", sum(src_counts.values()), "| per partition:", src_counts)

workers registered with the master
    host  cores  memory_MB state                   ui
10.8.0.6      2       2048 ALIVE http://10.8.0.6:8081
10.8.0.4      2       2048 ALIVE http://10.8.0.4:8082
10.8.0.5      2       2048 ALIVE http://10.8.0.5:8081
10.8.0.2      2       2048 ALIVE http://10.8.0.2:8081
10.8.0.3      2       2048  DEAD http://10.8.0.3:8081

alive workers: 4 | total cores: 8 | total memory: 8192 MB

source topic wiki.recentchange.raw: 6 partitions
source snapshot messages: 109868 | per partition: {0: 4429, 1: 4002, 2: 6575, 3: 34406, 4: 4555, 5: 55901}


## 2. Controlled corpus and benchmark topics

The real snapshot (≈110 k events) is read once from the driver, which sits next to the broker. It is cycled
to 200,000 messages; every cycled copy gets `id += cycle·10⁷` and `timestamp += cycle·3600 s`, so record
keys stay unique while the distribution of wikis, bots, titles and comments is preserved exactly.

Four balanced topics (`P = 1, 2, 4, 8`) receive the **same 200,000 payloads** in round-robin partition
order. A fifth topic keeps the original `wiki` keys at 6 partitions — it reproduces the skew that Q1a
warned about (`commonswiki` / `wikidatawiki` dominate) and is used to quantify what that skew costs.

In [3]:
# ============================== 3. Build corpus + benchmark topics ==============================
def read_source_snapshot():
    c = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": "bench-read-" + uuid.uuid4().hex[:6],
                  "enable.auto.commit": False, "auto.offset.reset": "earliest",
                  "fetch.max.bytes": 50_000_000})
    parts = sorted(c.list_topics(SOURCE, timeout=10).topics[SOURCE].partitions)
    tps = [TopicPartition(SOURCE, p) for p in parts]
    ends = {p: c.get_watermark_offsets(TopicPartition(SOURCE, p), timeout=5, cached=False)[1]
            for p in parts}
    c.assign(tps)
    total = sum(ends.values())
    out = []
    while len(out) < total:
        msgs = c.consume(5000, timeout=10)
        if not msgs:
            break
        out.extend(m.value() for m in msgs if m is not None and not m.error())
    c.close()
    return out


corpus = read_source_snapshot()
print("snapshot read:", len(corpus), "events, mean %.0f B/value" % (sum(map(len, corpus)) / len(corpus)))

payloads, keys = [], []
base = len(corpus)
for i in range(CORPUS_N):
    if i < base:
        payloads.append(corpus[i])
        keys.append(json.loads(corpus[i]).get("wiki") or "unknown")
    else:
        d = json.loads(corpus[i % base])
        cyc = i // base
        d["id"] = int(d.get("id") or 0) + cyc * 10_000_000
        d["timestamp"] = int(d.get("timestamp") or 0) + cyc * 3600
        payloads.append(json.dumps(d).encode())
        keys.append(d.get("wiki") or "unknown")
print("corpus built:", len(payloads), "payloads,", round(sum(map(len, payloads)) / 2**20, 1), "MiB")


def topic_counts(name):
    c = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": "cnt-" + uuid.uuid4().hex[:6]})
    try:
        n = len(c.list_topics(name, timeout=5).topics[name].partitions)
        counts = []
        for p in range(n):
            lo, hi = c.get_watermark_offsets(TopicPartition(name, p), timeout=5, cached=False)
            counts.append(hi - lo)
        return counts
    except Exception:
        return []
    finally:
        c.close()


def recreate_topic(name, partitions):
    with contextlib.suppress(Exception):
        for _, f in admin.delete_topics([name], operation_timeout=10).items():
            f.result()
        time.sleep(1)
    for _, f in admin.create_topics([NewTopic(name, num_partitions=partitions, replication_factor=1,
                                              config={"retention.ms": "3600000"})]).items():
        f.result()


def produce_topic(name, partitions, keyed=False):
    p = Producer({"bootstrap.servers": BOOTSTRAP, "linger.ms": 20, "batch.num.messages": 10000,
                  "batch.size": 262144, "acks": "1", "queue.buffering.max.messages": 200000})
    t0 = time.time()
    for i, v in enumerate(payloads):
        while True:
            try:
                if keyed:
                    p.produce(name, key=keys[i].encode(), value=v)
                else:
                    p.produce(name, value=v, partition=i % partitions)
                break
            except BufferError:
                p.poll(0.01)
    p.flush(120)
    return round(time.time() - t0, 2)


TOPICS = {}
produce_log = []
for P in P_LIST:
    name = "%s.p%d" % (PREFIX, P)
    counts = topic_counts(name)
    if len(counts) == P and all(counts) and sum(counts) == CORPUS_N:
        dt = 0.0
        print("topic %s already populated (%d msgs/partition)" % (name, counts[0]))
    else:
        recreate_topic(name, P)
        dt = produce_topic(name, P)
        counts = topic_counts(name)
    TOPICS[P] = name
    TOPIC_COUNTS[name] = counts
    produce_log.append({"topic": name, "partitions": P, "seconds": dt,
                        "msg_per_s": int(CORPUS_N / dt) if dt else None,
                        "counts": counts})

SKEW_TOPIC = "%s.skew6" % PREFIX
counts = topic_counts(SKEW_TOPIC)
if len(counts) == 6 and all(counts) and sum(counts) == CORPUS_N:
    dt = 0.0
    print("skew topic already populated:", counts)
else:
    recreate_topic(SKEW_TOPIC, 6)
    dt = produce_topic(SKEW_TOPIC, 6, keyed=True)
    counts = topic_counts(SKEW_TOPIC)
TOPIC_COUNTS[SKEW_TOPIC] = counts
produce_log.append({"topic": SKEW_TOPIC, "partitions": 6, "seconds": dt,
                    "msg_per_s": int(CORPUS_N / dt) if dt else None, "counts": counts})

print()
table(pd.DataFrame(produce_log), "topic production (same 200,000 payloads in each topic)")
for t, counts in TOPIC_COUNTS.items():
    print("%-18s P=%d  counts=%s  max/min=%.2f  Jain=%.3f"
          % (t, len(counts), counts, max(counts) / max(min(counts), 1), jain(counts)))

snapshot read: 109868 events, mean 357 B/value
corpus built: 200000 payloads, 72.5 MiB
topic wiki.bench.p1 already populated (200000 msgs/partition)
topic wiki.bench.p2 already populated (100000 msgs/partition)
topic wiki.bench.p4 already populated (50000 msgs/partition)
topic wiki.bench.p8 already populated (25000 msgs/partition)
skew topic already populated: [95550, 50559, 9921, 10641, 27706, 5623]

topic production (same 200,000 payloads in each topic)
           topic  partitions  seconds msg_per_s                                                   counts
   wiki.bench.p1           1      0.0      None                                                 [200000]
   wiki.bench.p2           2      0.0      None                                         [100000, 100000]
   wiki.bench.p4           4      0.0      None                             [50000, 50000, 50000, 50000]
   wiki.bench.p8           8      0.0      None [25000, 25000, 25000, 25000, 25000, 25000, 25000, 25000]
wiki.bench.skew

## 3. Kafka latency microbenchmark (produce → consume)

Before the Spark pipeline, the broker path itself is measured, because every stage latency below contains
it. All measurements run **on the driver** (same host as the broker, so this is broker + client cost, not
the lab LAN): a producer writes `N` messages carrying their own send timestamp, a consumer subscribed to the
same topic reads them back and the difference is the round-trip latency.

Three producer configurations are compared per partition count:

| id | configuration | what it tests |
|---|---|---|
| `A-acks-all-linger0` | `acks=all`, `linger.ms=0` | safest, no batching delay |
| `B-acks1-linger5` | `acks=1`, `linger.ms=5` | leader-only ack + small batching |
| `C-acksall-linger20-lz4` | `acks=all`, `linger.ms=20`, `compression=lz4` | batching + compression |

Producer throughput (50,000 messages) and consumer fan-out (1 vs 4 consumers over the 8-partition topic)
complete the microbenchmark.

In [5]:
# ============================== 4. Kafka microbenchmark ==============================
import multiprocessing as mp

MICRO_VARIANTS = {
    "A-acks-all-linger0":     {"acks": "all", "linger.ms": 0},
    "B-acks1-linger5":        {"acks": "1",   "linger.ms": 5},
    "C-acksall-linger20-lz4": {"acks": "all", "linger.ms": 20, "compression.type": "lz4"},
}

# latency topics mirror the matrix partition counts but are kept separate, so the matrix
# topics stay at exactly 200,000 messages
LAT_TOPICS = {}
for P in P_LIST:
    name = "%s.lat%d" % (PREFIX, P)
    if len(topic_counts(name)) != P:
        recreate_topic(name, P)
    LAT_TOPICS[P] = name


def kafka_roundtrip(topic, n=2000, cfg=None, timeout=60):
    p = Producer(dict({"bootstrap.servers": BOOTSTRAP}, **(cfg or {})))
    c = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": "lat-" + uuid.uuid4().hex[:8],
                  "auto.offset.reset": "latest", "enable.auto.commit": False,
                  "fetch.wait.max.ms": 20, "fetch.min.bytes": 1})
    c.subscribe([topic])
    t0 = time.time()
    while time.time() - t0 < 10 and not c.assignment():
        c.poll(0.2)
    for tp in c.assignment() or []:
        _, hi = c.get_watermark_offsets(tp, timeout=5, cached=False)
        c.seek(TopicPartition(tp.topic, tp.partition, hi))
    c.poll(0.5)
    for i in range(n):
        ts = time.time()
        p.produce(topic, key=str(i).encode(), value=json.dumps({"i": i, "t": ts}).encode())
    p.flush(timeout)
    lat, got = [], 0
    deadline = time.time() + timeout
    while got < n and time.time() < deadline:
        m = c.poll(0.5)
        if m is None or m.error():
            continue
        d = json.loads(m.value())
        lat.append((time.time() - d["t"]) * 1000.0)
        got += 1
    c.close()
    return lat, got


micro_rows = []
for P in P_LIST:
    for vname, cfg in MICRO_VARIANTS.items():
        lat, got = kafka_roundtrip(LAT_TOPICS[P], n=2000, cfg=cfg)
        micro_rows.append({"kind": "latency", "topic": LAT_TOPICS[P], "P": P, "variant": vname, "n": got,
                           "p50_ms": round(float(np.percentile(lat, 50)), 2) if lat else None,
                           "p95_ms": round(float(np.percentile(lat, 95)), 2) if lat else None,
                           "p99_ms": round(float(np.percentile(lat, 99)), 2) if lat else None,
                           "mean_ms": round(float(np.mean(lat)), 2) if lat else None})
    print("latency P=%d done" % P, flush=True)

# ---- producer throughput (throw-away topic) ----
THR_TOPIC = "%s.thr8" % PREFIX
if len(topic_counts(THR_TOPIC)) != 8:
    recreate_topic(THR_TOPIC, 8)
THR_N = 50_000
for vname, cfg in (("none-acks1", {"acks": "1", "linger.ms": 5}),
                   ("lz4-acks1", {"acks": "1", "linger.ms": 5, "compression.type": "lz4"}),
                   ("none-acksall", {"acks": "all", "linger.ms": 0})):
    p = Producer(dict({"bootstrap.servers": BOOTSTRAP, "batch.num.messages": 10000,
                       "batch.size": 262144, "queue.buffering.max.messages": 200000}, **cfg))
    t0 = time.time()
    for i in range(THR_N):
        while True:
            try:
                p.produce(THR_TOPIC, value=payloads[i % len(payloads)], partition=i % 8)
                break
            except BufferError:
                p.poll(0.01)
    p.flush(120)
    dt = time.time() - t0
    mb = sum(len(payloads[i % len(payloads)]) for i in range(THR_N)) / 2**20
    micro_rows.append({"kind": "producer_throughput", "topic": THR_TOPIC, "P": 8, "variant": vname,
                       "n": THR_N, "p50_ms": None, "p95_ms": None, "p99_ms": None,
                       "mean_ms": round(dt, 2), "msg_per_s": int(THR_N / dt),
                       "mb_per_s": round(mb / dt, 2)})
    print("producer throughput", vname, int(THR_N / dt), "msg/s", flush=True)


# ---- consumer fan-out: N independent processes, explicit partition assignment ----
def _fanout_worker(topic, parts, starts_json, ends_json, start_at, q):
    from confluent_kafka import Consumer, TopicPartition
    starts = {int(k): v for k, v in json.loads(starts_json).items()}
    ends = {int(k): v for k, v in json.loads(ends_json).items()}
    c = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": "fan-" + uuid.uuid4().hex[:8],
                  "enable.auto.commit": False, "fetch.max.bytes": 50_000_000})
    c.assign([TopicPartition(topic, p, starts[p]) for p in parts])
    while time.time() < start_at:
        time.sleep(0.02)
    t0 = time.time()
    got = 0
    while time.time() - t0 < 180:
        msgs = c.consume(5000, timeout=0.5)
        if msgs:
            got += sum(1 for m in msgs if m is not None and not m.error())
            continue
        pos = {tp.partition: c.position([tp])[0].offset for tp in c.assignment() or []}
        if pos and all(pos.get(p, 0) >= ends[p] for p in parts):
            break
    dt = time.time() - t0
    c.close()
    q.put({"pid": os.getpid(), "partitions": list(parts), "messages": got, "seconds": round(dt, 2)})


def fanout_run(nproc, topic):
    probe = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": "probe-" + uuid.uuid4().hex[:6]})
    md_ = probe.list_topics(topic, timeout=10).topics[topic]
    starts, ends = {}, {}
    for p in sorted(md_.partitions):
        lo, hi = probe.get_watermark_offsets(TopicPartition(topic, p), timeout=5, cached=False)
        starts[p], ends[p] = lo, hi
    probe.close()
    parts = sorted(starts)
    q = mp.Queue()
    start_at = time.time() + 3
    procs = [mp.Process(target=_fanout_worker,
                        args=(topic, parts[k::nproc], json.dumps(starts), json.dumps(ends), start_at, q))
             for k in range(nproc)]
    for pr in procs:
        pr.start()
    for pr in procs:
        pr.join(240)
    rows = []
    for _ in procs:
        try:
            rows.append(q.get(timeout=30))
        except Exception:
            rows.append({"partitions": [], "messages": 0, "seconds": 1.0})
    total = sum(r["messages"] for r in rows)
    dt = max(r["seconds"] for r in rows) or 1.0
    return {"kind": "consumer_fanout", "consumers": nproc, "messages": total,
            "seconds": round(dt, 2), "msg_per_s": int(total / dt),
            "per_consumer": [r["messages"] for r in rows],
            "partitions_per_consumer": [r["partitions"] for r in rows]}


fanout_rows = []
for ncons in (1, 4):
    row = fanout_run(ncons, TOPICS[8])
    fanout_rows.append(row)
    print("fan-out", ncons, "consumer(s):", row["messages"], "msgs in", row["seconds"], "s ->",
          row["msg_per_s"], "msg/s", flush=True)

micro_df = pd.DataFrame(micro_rows)
fanout_df = pd.DataFrame(fanout_rows)
MICRO_JSON.write_text(json.dumps({"rows": micro_rows, "fanout": fanout_rows}, indent=1))
table(micro_df[micro_df.kind == "latency"]
      .pivot_table(index="P", columns="variant", values="p99_ms").round(2),
      "Kafka produce->consume p99 latency (ms)")
table(micro_df[micro_df.kind == "producer_throughput"], "producer throughput")
table(fanout_df, "consumer fan-out (explicit partition assignment, one process per slice)")

latency P=1 done
latency P=2 done
latency P=4 done
latency P=8 done
producer throughput none-acks1 608681 msg/s
producer throughput lz4-acks1 1108542 msg/s
producer throughput none-acksall 748597 msg/s
fan-out 1 consumer(s): 200000 msgs in 0.71 s -> 281690 msg/s
fan-out 4 consumer(s): 200000 msgs in 0.54 s -> 370370 msg/s
Kafka produce->consume p99 latency (ms)
 A-acks-all-linger0  B-acks1-linger5  C-acksall-linger20-lz4
              37.81            14.71                   14.78
              71.17            21.11                   21.29
              39.85            14.16                   25.72
              79.67            17.08                   24.24

producer throughput
               kind           topic  P      variant     n  p50_ms  p95_ms  p99_ms  mean_ms  msg_per_s  mb_per_s
producer_throughput wiki.bench.thr8  8   none-acks1 50000     NaN     NaN     NaN     0.08   608681.0    202.04
producer_throughput wiki.bench.thr8  8    lz4-acks1 50000     NaN     NaN     NaN     

,kind,consumers,messages,seconds,msg_per_s,per_consumer,partitions_per_consumer
0,consumer_fanout,1,200000,0.71,281690,[200000],"[[0, 1, 2, 3, 4, 5, 6, 7]]"
1,consumer_fanout,4,200000,0.54,370370,"[50000, 50000, 50000, 50000]","[[1, 5], [3, 7], [0, 4], [2, 6]]"


### Kafka microbenchmark — how to read it

*Latency* grows with `linger.ms` (batching) but batching is what buys throughput; `acks=all` adds a
replication round trip. *Partition count* changes the broker's parallelism ceiling, not the latency of a
single message. The fan-out table shows the consumer side: one consumer is limited by its single-threaded
poll loop, four consumers in the same group split the eight partitions and should scale close to 4× — until
the driver's CPU or the broker's disk becomes the limit.

In [5]:
# ============================== 5. Kafka microbenchmark charts ==============================
CH = {}
def save(fig, name):
    p = PLOT_DIR / (name + ".png")
    fig.tight_layout()
    fig.savefig(p, dpi=110)
    CH[name] = p
    plt.show()
    plt.close(fig)


lat = micro_df[micro_df.kind == "latency"]
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for a, col in zip(ax, ("p50_ms", "p95_ms", "p99_ms")):
    sns.barplot(data=lat, x="P", y=col, hue="variant", ax=a)
    a.set(title=col.replace("_ms", " latency"), xlabel="Kafka partitions", ylabel="ms")
    a.legend(fontsize=7, title=None)
fig.suptitle("Kafka produce->consume latency by partition count and producer configuration")
save(fig, "k1_kafka_latency")

thr = micro_df[micro_df.kind == "producer_throughput"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(data=thr, x="variant", y="msg_per_s", ax=ax[0], color="#3182ce")
ax[0].set(title="Producer throughput", xlabel="", ylabel="messages / s")
sns.barplot(data=fanout_df, x="consumers", y="msg_per_s", ax=ax[1], color="#38a169")
ax[1].set(title="Consumer fan-out (8 partitions, 200k messages)", xlabel="consumers in group",
          ylabel="messages / s")
for a in ax:
    plt.setp(a.get_xticklabels(), rotation=12, ha="right")
save(fig, "k2_kafka_throughput")
print("kafka charts:", [k for k in CH])

kafka charts: ['k1_kafka_latency', 'k2_kafka_throughput']


## 4. Spark pipeline and the P × W matrix

Each cell of the matrix is a **fresh Spark application** with `spark.cores.max = 2·W` (2 cores per executor,
1200 MB). Inside an application:

1. wait until the requested executors register, then run a discarded warm-up job — executors fetch the
   `spark-sql-kafka` connector jars on first use, and on this cluster that cold start is 30–90 s; it is
   recorded as `cold_start_s` and excluded from the stage times;
2. for every P, read the P-partition topic, `persist()` it, and measure the six stages below with
   `measure()`;
3. after each stage, pull per-executor task counts and CPU time from the REST API → weight distribution;
4. sample the driver's CPU/RAM/network while the stages run → resources.

A run's `makespan` is the sum of its stage wall times; `throughput = 200,000 / makespan`.

In [6]:
# ============================== 6. Pipeline stages ==============================
from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, StringType, LongType, IntegerType, BooleanType)

EVENT_SCHEMA = StructType([
    StructField("id", LongType()), StructField("type", StringType()), StructField("wiki", StringType()),
    StructField("server_name", StringType()), StructField("title", StringType()),
    StructField("namespace", IntegerType()), StructField("user", StringType()),
    StructField("bot", BooleanType()), StructField("minor", BooleanType()),
    StructField("timestamp", LongType()), StructField("comment", StringType()),
    StructField("length", StructType([StructField("old", LongType()), StructField("new", LongType())])),
])

STOPWORDS = set("""yang dan di ke dari ini itu dengan untuk pada adalah tidak akan juga dalam saya kami kita
dia mereka ada bisa sudah belum karena agar oleh sebagai atau jika maka saat setelah sebelum lebih paling
sangat hanya saja para semua beberapa satu dua tiga empat lima hal cara orang tahun baru lama masih pernah
sedang telah harus ingin mau tapi tetapi namun sehingga bahwa antara tanpa hingga sampai ketika sebab kalau
pun lah kah nya ku mu se secara kepada terhadap merupakan menjadi dibuat diubah ditambah dihapus termasuk
berikut yaitu yakni sini sana""".split())

WORD_RE = re.compile(r"[^\W\d_]+", re.UNICODE)


def parse_line(value, fail_acc, ok_acc):
    try:
        d = json.loads(value.decode("utf-8"))
        length = d.get("length") or {}
        rec = (d.get("wiki"), d.get("type"), d.get("title"), bool(d.get("bot")), d.get("user"),
               d.get("timestamp"), length.get("old"), length.get("new"), d.get("namespace"),
               d.get("comment") or "")
        if not rec[0] or not rec[1]:
            raise ValueError("no wiki/type")
        ok_acc.add(1)
        return rec
    except Exception:
        fail_acc.add(1)
        return None


def new_session(W, tag):
    s = (SparkSession.builder
         .appName("wiki-bench-W%d-%s" % (W, tag))
         .config("spark.cores.max", str(2 * W))
         .config("spark.executor.cores", "2")
         .config("spark.executor.memory", "512m")
         .config("spark.python.worker.memory", "128m")
         .config("spark.sql.shuffle.partitions", "16")
         .config("spark.sql.adaptive.enabled", "true")
         .getOrCreate())
    s.sparkContext.setLogLevel("WARN")
    return s


def wait_and_warmup(spark, W, rounds=3):
    """Wait for executors, then force them through their first task (jar fetch + python worker)."""
    sc = spark.sparkContext
    avail = sum(w["cores"] for w in http_json(MASTER_UI + "/json/")["workers"] if w["state"] == "ALIVE")
    want = max(1, min(W, avail // 2))
    t0 = time.time()
    while time.time() - t0 < 120:
        n = len(sc._jsc.sc().getExecutorMemoryStatus().keySet().toString().split(", ")) - 1
        if n >= want:
            break
        time.sleep(3)
    reg = len(sc._jsc.sc().getExecutorMemoryStatus().keySet().toString().split(", ")) - 1

    def work(i, it):
        n = sum(1 for _ in it)
        return [(socket.gethostname(), n)]

    for _ in range(rounds):
        sc.parallelize(range(16), 16).mapPartitionsWithIndex(work).collect()
    active = [e for e in ui_json(sc, "/executors")
              if e["id"] != "driver" and e.get("totalTasks", 0) > 0]
    return {"cold_start_s": round(time.time() - t0, 1), "executors_registered": reg,
            "executors_wanted": want, "cluster_cores": avail, "executors_active": len(active),
            "executor_hosts": sorted({e["hostPort"].split(":")[0] for e in active})}


def exec_snapshot(sc):
    return {e["id"]: [e.get("totalTasks", 0), e.get("totalDuration", 0)]
            for e in ui_json(sc, "/executors") if e["id"] != "driver"}


def run_pipeline(spark, topic, P, W, part_counts):
    """Six measured stages over one topic. Returns one result record."""
    sc = spark.sparkContext
    del STAGE_LOG[:]
    pm = ResourceSampler().start()
    rec = {"W": W, "P": P, "topic": topic, "started": datetime.now().strftime("%H:%M:%S"),
           "jain_partition": jain(part_counts)}
    kdf = parsed = None
    before = exec_snapshot(sc)
    try:
        kdf = (spark.read.format("kafka")
               .option("kafka.bootstrap.servers", BOOTSTRAP)
               .option("subscribe", topic)
               .option("startingOffsets", "earliest")
               .option("endingOffsets", "latest")
               .load()
               .select("value", "partition", "offset", "timestamp"))
        kdf.persist(StorageLevel.DISK_ONLY)     # keep executor heap small: the worker PCs have 2 GB
        rec["S1_ingest"] = measure(sc, "S1_ingest", lambda: kdf.count())        # I/O + cache
        fail = sc.accumulator(0)
        ok = sc.accumulator(0)
        parsed = (kdf.select("value").rdd
                  .map(lambda r: parse_line(r[0], fail, ok))
                  .filter(lambda x: x is not None)
                  .persist(StorageLevel.DISK_ONLY))
        rec["S2_parse"] = measure(sc, "S2_parse", lambda: parsed.count())      # Q2a + cache
        rec["parse_failures"] = fail.value
        rec["parsed_ok"] = ok.value

        add = lambda a, b: a + b
        rec["S3_reduceByKey"] = measure(
            sc, "S3_reduceByKey",
            lambda: parsed.filter(lambda r: r[1] == "edit" and r[3] is False)
                          .map(lambda r: (r[0], 1)).reduceByKey(add)
                          .takeOrdered(10, key=lambda kv: (-kv[1], kv[0])))
        rec["S4_groupByKey"] = measure(
            sc, "S4_groupByKey",
            lambda: parsed.filter(lambda r: r[1] == "edit" and r[3] is False)
                          .map(lambda r: (r[0], 1)).groupByKey().mapValues(sum)
                          .takeOrdered(10, key=lambda kv: (-kv[1], kv[0])))

        edf = (kdf.select(F.from_json(F.col("value").cast("string"), EVENT_SCHEMA).alias("e"))
                  .select("e.*")
                  .withColumn("event_ts", F.to_timestamp(F.from_unixtime("timestamp")))
                  .withColumn("selisih_byte",
                              F.col("length.new") - F.coalesce(F.col("length.old"), F.lit(0))))
        edf.createOrReplaceTempView("wiki_edits")
        rec["S5_sql_aggregate"] = measure(
            sc, "S5_sql_aggregate",
            lambda: spark.sql("SELECT wiki, COUNT(*) edits, "
                              "SUM(CASE WHEN bot THEN 1 ELSE 0 END) bot_edits "
                              "FROM wiki_edits WHERE type='edit' GROUP BY wiki "
                              "ORDER BY edits DESC").collect())
        rec["S6_sql_window"] = measure(
            sc, "S6_sql_window",
            lambda: spark.sql("SELECT COUNT(*) AS spikes FROM ("
                              "SELECT wiki, title, window(event_ts,'5 minutes') w, COUNT(*) c "
                              "FROM wiki_edits WHERE type='edit' AND title IS NOT NULL "
                              "GROUP BY wiki, title, window(event_ts,'5 minutes') "
                              "HAVING COUNT(*) >= 5)").first()["spikes"])

        rec["stages"] = list(STAGE_LOG)
        rec["makespan"] = round(sum(s["seconds"] for s in STAGE_LOG), 3)
        rec["throughput_msg_s"] = int(CORPUS_N / rec["makespan"])
        rec["message_count"] = int(rec["S1_ingest"])
        rec["shuffle_write"] = sum(s["shuffle_write"] for s in STAGE_LOG)
        rec["shuffle_read"] = sum(s["shuffle_read"] for s in STAGE_LOG)
        rec["spilled"] = sum(s["spilled"] for s in STAGE_LOG)

        ex = [e for e in ui_json(sc, "/executors") if e["id"] != "driver"]
        rec["executors"] = [{"id": e["id"], "host": e["hostPort"].split(":")[0], "cores": e["totalCores"],
                             "tasks": e.get("totalTasks", 0) - before.get(e["id"], [0, 0])[0],
                             "duration_ms": e.get("totalDuration", 0) - before.get(e["id"], [0, 0])[1],
                             "gc_ms": e.get("totalGCTime", 0),
                             "memory_used_mb": round(e.get("memoryUsed", 0) / 2**20, 1),
                             "memory_max_mb": round(e.get("maxMemory", 0) / 2**20, 1)} for e in ex]
        per_exec_tasks = [e["tasks"] for e in rec["executors"]]
        rec["jain_exec"] = jain(per_exec_tasks + [0] * max(0, W - len(per_exec_tasks)))
        rec["active_executors"] = sum(1 for e in rec["executors"] if e["tasks"] > 0)
        core_ms = sum(e["duration_ms"] for e in rec["executors"])
        rec["core_util"] = (round(core_ms / (rec["active_executors"] * 2 * 1000 * rec["makespan"]), 3)
                            if rec["active_executors"] else None)
        rec["exec_mem_mb"] = round(sum(e["memory_used_mb"] for e in rec["executors"]), 1)
        rec["gc_s"] = round(sum(e["gc_ms"] for e in rec["executors"]) / 1000, 2)
    except Exception as e:
        rec["error"] = "%s: %s" % (type(e).__name__, e)
        with contextlib.suppress(Exception):
            rec["stages"] = list(STAGE_LOG)
    finally:
        rec["resources"] = pm.stop()
        if parsed is not None:
            with contextlib.suppress(Exception):
                parsed.unpersist()
        if kdf is not None:
            with contextlib.suppress(Exception):
                kdf.unpersist()
    return rec


print("pipeline helpers ready")

pipeline helpers ready


## 5. Running the matrix

3 applications (W) × 4 topics (P) = 12 runs, plus one skew run at W=2. A warm-up application is executed
first and thrown away, and each application runs its own internal warm-up, so the reported numbers are
steady-state. Every result is appended to `bench_results.json` as soon as the run finishes, so the cell can
be re-executed and will skip combinations that already succeeded in this session.

Expected duration: **10–15 minutes** — the dominant cost is the 30–90 s executor cold start per application.

In [7]:
# ============================== 7. Matrix runner ==============================
SESSION = STARTED.strftime("%Y%m%d")
R = []
if RESULT_JSON.exists():
    with contextlib.suppress(Exception):
        R = [r for r in json.loads(RESULT_JSON.read_text()) if r.get("session") == SESSION]
done = {(r["W"], r["P"]) for r in R if not r.get("warmup") and "error" not in r and r.get("variant") != "skew"}
print("runs already recorded in this session:", sorted(done))

# --- discarded warm-up application -------------------------------------------------
if not any(r.get("warmup") for r in R):
    sp = new_session(1, "warm")
    warm = wait_and_warmup(sp, 1, rounds=1)
    print("warm-up application:", warm)
    R.append({"warmup": True, "session": SESSION, "W": 1, "P": 0, "cold_start": warm})
    RESULT_JSON.write_text(json.dumps(R, indent=1))
    sp.stop()
    time.sleep(5)
else:
    print("warm-up already recorded")


def run_group(W, todo, attempt):
    sp = new_session(W, "a%d" % attempt)
    sc = sp.sparkContext
    info = wait_and_warmup(sp, W, rounds=2)
    print("\n=== W=%d (attempt %d) app %s | %s" % (W, attempt, sc.applicationId, info), flush=True)
    ok = []
    for P in todo:
        r = run_pipeline(sp, TOPICS[P], P, W, TOPIC_COUNTS[TOPICS[P]])
        r.update({"requested_workers": W, "session": SESSION, "cold_start": info,
                  "application": sc.applicationId, "driver_ui": sc.uiWebUrl})
        R.append(r)
        RESULT_JSON.write_text(json.dumps(R, indent=1))
        if "error" in r:
            print("  P=%d FAILED: %s" % (P, r["error"]), flush=True)
            break                      # the application usually lost its executors - retry in a new one
        else:
            ok.append(P)
            print("  P=%d makespan=%.1fs throughput=%d msg/s active_exec=%d jain=%.2f util=%.2f"
                  % (P, r["makespan"], r["throughput_msg_s"], r["active_executors"],
                     r["jain_exec"], r["core_util"] or 0), flush=True)
    sp.stop()
    time.sleep(5)
    return ok


for attempt in (1, 2):
    for W in W_LIST:
        todo = [P for P in P_LIST if (W, P) not in done]
        if not todo:
            print("W=%d complete, skipping" % W, flush=True)
            continue
        for P in run_group(W, todo, attempt):
            done.add((W, P))
    if all((W, P) in done for W in W_LIST for P in P_LIST):
        break
    print("\n--- retrying failed combinations (attempt %d done) ---" % attempt, flush=True)

# --- skew comparison at fixed W=2 (wiki-keyed topic, 6 partitions) ------------------
if not any(r.get("variant") == "skew" and "error" not in r for r in R):
    sp = new_session(2, "skew")
    info = wait_and_warmup(sp, 2, rounds=2)
    r = run_pipeline(sp, SKEW_TOPIC, 6, 2, TOPIC_COUNTS[SKEW_TOPIC])
    r.update({"requested_workers": 2, "session": SESSION, "cold_start": info, "variant": "skew",
              "application": sp.sparkContext.applicationId, "driver_ui": sp.sparkContext.uiWebUrl})
    R.append(r)
    RESULT_JSON.write_text(json.dumps(R, indent=1))
    print("\nskew topic:", "OK" if "error" not in r else r["error"],
          "| jain_partition=%.2f" % r.get("jain_partition", 0), flush=True)
    sp.stop()

print("\nmatrix complete: %d of %d combinations recorded"
      % (len(done), len(W_LIST) * len(P_LIST)))

runs already recorded in this session: [(1, 1), (1, 2), (1, 4), (1, 8), (2, 1), (2, 2), (2, 4), (2, 8), (4, 1), (4, 2), (4, 4), (4, 8)]
warm-up already recorded
W=1 complete, skipping
W=2 complete, skipping
W=4 complete, skipping

matrix complete: 12 of 12 combinations recorded


## 6. Results

The tables below are computed from `bench_results.json`; the numbers in the HTML report are the same.

In [8]:
# ============================== 8. Result tables ==============================
runs = [r for r in R if not r.get("warmup") and "error" not in r]
warm = next((r for r in R if r.get("warmup")), None)
bad = [r for r in R if r.get("error")]

rows = []
for r in runs:
    rows.append({
        "variant": r.get("variant", "balanced"), "P": r["P"], "W": r["W"],
        "makespan_s": r["makespan"], "throughput_msg_s": r["throughput_msg_s"],
        "messages": r.get("message_count"),
        "active_exec": r["active_executors"], "exec_reg": r["cold_start"]["executors_registered"],
        "hosts": ",".join(sorted({e["host"] for e in r.get("executors", []) if e["tasks"] > 0})) or "-",
        "cold_start_s": r["cold_start"]["cold_start_s"],
        "jain_exec": r["jain_exec"], "jain_partition": r.get("jain_partition"),
        "core_util": r["core_util"],
        "driver_cpu_pct": r["resources"]["cpu_mean"], "net_rx_mb": r["resources"]["net_rx_mb"],
        "exec_mem_mb": r["exec_mem_mb"], "gc_s": r["gc_s"],
        "shuffle_wr_mb": round(r["shuffle_write"] / 2**20, 1),
        "shuffle_rd_mb": round(r["shuffle_read"] / 2**20, 1),
    })
res = pd.DataFrame(rows).sort_values(["variant", "P", "W"])
table(res, "all runs")

bal = res[res.variant == "balanced"].copy()
if not bal.empty:
    base = bal[bal.W == bal.W.min()].set_index("P")["makespan_s"]
    bal["speedup"] = [base.get(p, np.nan) / t for p, t in zip(bal.P, bal.makespan_s)]
    bal["efficiency"] = bal.speedup / bal.active_exec.replace(0, np.nan)
    bal["efficiency_req"] = bal.speedup / bal.W
    # a stage can only use min(P, active executors) readers on the ingest side
    bal["parallel_limit"] = [min(p, a) for p, a in zip(bal.P, bal.active_exec)]
    bal["eff_vs_limit"] = bal.speedup / bal.parallel_limit.replace(0, np.nan)
    table(bal[["P", "W", "hosts", "makespan_s", "throughput_msg_s", "speedup", "efficiency",
               "eff_vs_limit", "active_exec", "jain_exec", "core_util"]],
          "balanced runs: speedup and efficiency (baseline W=%d)" % bal.W.min())
    table(bal.pivot_table(index="P", columns="W", values="throughput_msg_s").round(0),
          "throughput (msg/s) — P × W")
    table(bal.pivot_table(index="P", columns="W", values="speedup").round(2), "speedup vs W=1")
    table(bal.pivot_table(index="P", columns="W", values="efficiency").round(2), "parallel efficiency")
    table(bal.pivot_table(index="P", columns="W", values="core_util").round(2), "cluster core utilisation")
    # Little's law: L = throughput x mean time a message spends in the pipeline
    bal["little_L"] = (bal.throughput_msg_s * bal.makespan_s / 6).round(1)  # 6 stages -> mean stage latency
    table(bal[["P", "W", "throughput_msg_s", "makespan_s", "little_L"]].round(2),
          "Little's law check (messages in flight = throughput × mean stage latency)")

srows = []
for r in runs:
    for s in r.get("stages", []):
        srows.append({"variant": r.get("variant", "balanced"), "P": r["P"], "W": r["W"],
                      "stage": s["label"].split("_", 1)[0] + " " + s["label"].split("_", 1)[1].replace("_", " "),
                      "seconds": s["seconds"], "tasks": s["tasks"],
                      "p50_ms": s["task_p50_ms"], "p95_ms": s["task_p95_ms"],
                      "shuffle_wr_mb": round(s["shuffle_write"] / 2**20, 2),
                      "shuffle_rd_mb": round(s["shuffle_read"] / 2**20, 2),
                      "executors_used": len([k for k, v in s["by_executor"].items() if v])})
stages = pd.DataFrame(srows)
if not stages.empty:
    table(stages[stages.variant == "balanced"]
          .pivot_table(index=["P", "W"], columns="stage", values="seconds").round(2),
          "stage wall time (s) per run")
    table(stages[stages.variant == "balanced"].groupby("stage")[["seconds", "p50_ms", "p95_ms"]]
          .mean().round(3).reset_index(), "mean per stage over all balanced runs")

if (res.variant == "skew").any():
    table(res[res.variant == "skew"], "wiki-keyed (skewed) topic run")
if bad:
    table(pd.DataFrame([{"W": b.get("W"), "P": b.get("P"), "error": b["error"]} for b in bad]), "failed runs")

# publish the raw artefacts next to the notebook (downloadable through the Jupyter contents API)
import shutil
for src in (RESULT_JSON, MICRO_JSON):
    if Path(src).exists():
        shutil.copy(src, Path.cwd() / Path(src).name)
print("published:", [p.name for p in Path.cwd().glob("bench_results.json")] +
      [p.name for p in Path.cwd().glob("kafka_micro.json")])

all runs
 variant  P  W  makespan_s  throughput_msg_s  messages  active_exec  exec_reg             hosts  cold_start_s  jain_exec  jain_partition  core_util  driver_cpu_pct  net_rx_mb  exec_mem_mb  gc_s  shuffle_wr_mb  shuffle_rd_mb
balanced  1  1     256.970               778    200000            1         1          10.8.0.5          81.6      1.000           1.000      0.505             5.4        7.7          0.5  1.03            2.9            2.9
balanced  1  2     100.017              1999    200000            1         2          10.8.0.4          39.5      0.500           1.000      0.508             5.4        8.3          0.5  0.63            2.9            2.9
balanced  1  4     118.373              1689    200000            1         4          10.8.0.4          40.7      0.250           1.000      0.507             5.3       12.4          0.4  0.53            2.9            2.9
balanced  2  1     125.804              1589    200000            1         1          10.8.0.5

## 7. Charts

In [9]:
# ============================== 9. Charts ==============================
bal = res[res.variant == "balanced"].copy()
if not bal.empty:
    base = bal[bal.W == bal.W.min()].set_index("P")["makespan_s"]
    bal["speedup"] = [base.get(p, np.nan) / t for p, t in zip(bal.P, bal.makespan_s)]
    bal["efficiency"] = bal.speedup / bal.active_exec.replace(0, np.nan)
    Ps, Ws = sorted(bal.P.unique()), sorted(bal.W.unique())
    colors = dict(zip(Ps, sns.color_palette("tab10", len(Ps))))

    fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
    for P in Ps:
        s = bal[bal.P == P].sort_values("W")
        ax[0].plot(s.W, s.throughput_msg_s, "-o", color=colors[P], label="P=%d" % P)
        ax[1].plot(s.W, s.speedup, "-o", color=colors[P], label="P=%d" % P)
    ax[1].plot(Ws, Ws, "k--", lw=1, label="ideal")
    ax[0].set(title="Throughput vs workers", xlabel="active executors (W)", ylabel="messages / s")
    ax[1].set(title="Speedup vs W=1 (same P)", xlabel="W", ylabel="×")
    for a in ax[:2]:
        a.set_xticks(Ws)
        a.legend(fontsize=8)
    sns.heatmap(bal.pivot_table(index="P", columns="W", values="throughput_msg_s"),
                annot=True, fmt=".0f", cmap="YlGn", ax=ax[2], cbar_kws={"label": "msg/s"})
    ax[2].set(title="Throughput heatmap (msg/s)")
    save(fig, "c1_scalability")

    fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
    for P in Ps:
        s = bal[bal.P == P].sort_values("W")
        ax[0].plot(s.W, s.jain_exec, "-o", color=colors[P], label="P=%d" % P)
        ax[1].plot(s.W, s.core_util, "-o", color=colors[P], label="P=%d" % P)
        ax[2].plot(s.W, s.exec_mem_mb, "-o", color=colors[P], label="P=%d" % P)
    ax[0].set(title="Load balance across executors (Jain)", xlabel="W", ylabel="J (1 = even)", ylim=(0, 1.05))
    ax[1].set(title="Cluster core utilisation", xlabel="W", ylabel="task CPU / wall")
    ax[2].set(title="Executor memory in use", xlabel="W", ylabel="MiB")
    for a in ax:
        a.set_xticks(Ws)
        a.legend(fontsize=8)
    save(fig, "c2_balance_resource")

if not stages.empty:
    sb = stages[stages.variant == "balanced"]
    piv = sb.pivot_table(index=["P", "W"], columns="stage", values="seconds")
    fig, ax = plt.subplots(1, 2, figsize=(16, 4.8))
    bottom = np.zeros(len(piv))
    for col in piv.columns:
        vals = np.nan_to_num(piv[col].values)
        ax[0].bar(range(len(piv)), vals, bottom=bottom, label=col)
        bottom += vals
    ax[0].set_xticks(range(len(piv)))
    ax[0].set_xticklabels(["P%d-W%d" % x for x in piv.index], rotation=60, ha="right")
    ax[0].set(title="Stage wall time (stacked)", ylabel="seconds")
    ax[0].legend(fontsize=7, title=None)
    sns.barplot(data=sb, x="stage", y="p95_ms", hue="W", ax=ax[1])
    ax[1].set(title="Per-task p95 by stage and W", xlabel="", ylabel="ms")
    plt.setp(ax[1].get_xticklabels(), rotation=20, ha="right")
    ax[1].legend(fontsize=8, title="W")
    save(fig, "c3_stage_latency")

    fig, ax = plt.subplots(1, 2, figsize=(16, 4.4))
    wide = sb[sb.W == sb.W.max()]
    mat = wide.pivot_table(index=["P", "W"], columns="stage", values="tasks").fillna(0)
    if len(mat):
        sns.heatmap(mat, annot=True, fmt=".0f", cmap="Blues", ax=ax[0], cbar_kws={"label": "tasks"})
        ax[0].set(title="Tasks per stage at W=%d" % sb.W.max())
    ex_rows = []
    for r in runs:
        if r["W"] != max(Ws):
            continue
        tot = sum(e["tasks"] for e in r["executors"]) or 1
        for e in r["executors"]:
            ex_rows.append({"run": "P%d-W%d" % (r["P"], r["W"]), "executor": e["id"],
                            "host": e["host"], "share_pct": 100.0 * e["tasks"] / tot,
                            "cpu_s": e["duration_ms"] / 1000.0})
    if ex_rows:
        exdf = pd.DataFrame(ex_rows)
        sns.barplot(data=exdf, x="run", y="share_pct", hue="executor", ax=ax[1])
        ax[1].set(title="Executor share of tasks (W=%d)" % max(Ws), xlabel="", ylabel="% of tasks")
        ax[1].legend(fontsize=7, title=None)
    save(fig, "c4_weight_distribution")

# partition weights: balanced round-robin vs wiki-keyed
if TOPIC_COUNTS:
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    for i, (t, counts) in enumerate(sorted(TOPIC_COUNTS.items())):
        if "skew" in t:
            continue
        ax[0].bar([p + i * 0.2 for p in range(len(counts))], counts, width=0.2, label=t.split(".")[-1])
    ax[0].set(title="Messages per partition — balanced round-robin topics", xlabel="partition",
              ylabel="messages")
    ax[0].legend(fontsize=7)
    skew = [(t, c) for t, c in TOPIC_COUNTS.items() if "skew" in t]
    if skew:
        t, counts = skew[0]
        ax[1].bar(range(len(counts)), counts, color="#e53e3e")
        ax[1].set(title="wiki-keyed topic %s: max/min = %.1f×" % (t.split(".")[-1],
                                                                   max(counts) / max(min(counts), 1)),
                  xlabel="partition", ylabel="messages")
    save(fig, "c5_partition_skew")

# resources per run
if not bal.empty:
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    lab = ["P%d-W%d" % (p, w) for p, w in zip(bal.P, bal.W)]
    ax[0].bar(lab, bal.driver_cpu_pct, color="#4299e1")
    ax[0].set(title="Driver (broker host) CPU during the run", ylabel="% of 16 cores")
    ax[1].bar(lab, bal.net_rx_mb, color="#48bb78")
    ax[1].set(title="Network received by the driver", ylabel="MiB")
    ax[2].bar(lab, bal.gc_s, color="#ed8936")
    ax[2].set(title="Executor GC time", ylabel="seconds")
    for a in ax:
        plt.setp(a.get_xticklabels(), rotation=60, ha="right")
    save(fig, "c6_resources")

print("charts:", sorted(CH))

charts: ['c1_scalability', 'c2_balance_resource', 'c3_stage_latency', 'c4_weight_distribution', 'c5_partition_skew', 'c6_resources', 'k1_kafka_latency', 'k2_kafka_throughput']


## 8. HTML report

Self-contained: tables are rendered inline and every chart is embedded as base64, so the file can be
submitted as evidence without the notebook or the cluster.

In [10]:
# ============================== 10. HTML report ==============================
def img64(p):
    return base64.b64encode(Path(p).read_bytes()).decode()


def df_html(df, cols=None):
    d = df.copy()
    if cols:
        d = d[cols]
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].map(lambda x: "" if pd.isna(x) else "{:,.2f}".format(x))
    return d.to_html(index=False, border=0, classes="t")


def build_report():
    runs_ok = [r for r in R if not r.get("warmup") and "error" not in r]
    b = bal.copy()
    if b.empty:
        raise RuntimeError("no successful runs — run the matrix first")
    base = b[b.W == b.W.min()].set_index("P")["makespan_s"]
    b["speedup"] = [base.get(p, np.nan) / t for p, t in zip(b.P, b.makespan_s)]
    b["efficiency"] = b.speedup / b.active_exec.replace(0, np.nan)

    best = b.sort_values("throughput_msg_s").iloc[-1]
    wmax = b.W.max()
    at_max = b[b.W == wmax].sort_values("throughput_msg_s")
    best_max = at_max.iloc[-1]
    base_same = b[(b.P == best_max.P) & (b.W == b.W.min())]
    sp_max = (base_same.iloc[0].makespan_s / best_max.makespan_s) if len(base_same) else np.nan
    eff_max = sp_max / max(best_max.active_exec, 1) if pd.notna(sp_max) else np.nan

    lats = micro_df[micro_df.kind == "latency"] if "micro_df" in globals() else pd.DataFrame()
    thr = micro_df[micro_df.kind == "producer_throughput"] if "micro_df" in globals() else pd.DataFrame()
    fan = fanout_df if "fanout_df" in globals() else pd.DataFrame()
    skew_run = next((r for r in runs_ok if r.get("variant") == "skew"), None)

    bullets = [
        "Largest balanced configuration tested: <b>P=%d, W=%d</b> — <b>%s msg/s</b>, %.1f s for %s events."
        % (best.P, best.W, format(int(best.throughput_msg_s), ","), best.makespan_s,
           format(int(best["messages"]), ",")),
        "Scaling W=%d→%d at P=%d: speedup <b>%.2f×</b>, efficiency <b>%.2f</b> over %d active executor(s)."
        % (b.W.min(), wmax, best_max.P, sp_max, eff_max, best_max.active_exec),
    ]
    if len(at_max):
        bullets.append("At W=%d the partition count decides throughput: %s."
                       % (wmax, "; ".join("P=%d → %s msg/s" % (r.P, format(int(r.throughput_msg_s), ","))
                                          for _, r in at_max.iterrows())))
    if skew_run is not None:
        bullets.append("Wiki-keyed topic (Q1a skew, P=6, W=2): <b>%s msg/s</b> — partition Jain %.2f vs 1.00 for the "
                       "round-robin topics, so the key choice costs %.0f%% of the throughput."
                       % (format(int(skew_run["throughput_msg_s"]), ","), skew_run.get("jain_partition", 0),
                          100 * (1 - skew_run["throughput_msg_s"] /
                                 max(b[b.P == 8].throughput_msg_s.max(), 1))))
    if len(lats):
        bullets.append("Kafka produce→consume: p50 %.1f–%.1f ms, p99 up to <b>%.1f ms</b> across partition counts and "
                       "producer configurations." % (lats.p50_ms.min(), lats.p50_ms.max(), lats.p99_ms.max()))
    if len(fan) and set(fan.consumers) >= {1, 4}:
        f1 = fan[fan.consumers == 1].msg_per_s.iloc[0]
        f4 = fan[fan.consumers == 4].msg_per_s.iloc[0]
        bullets.append("Consumer fan-out: 1 → 4 consumers on the 8-partition topic: %s → %s msg/s (%.1f×)."
                       % (format(int(f1), ","), format(int(f4), ","), f4 / f1))

    parts = [
        "<h1>Wikipedia edit pipeline — Kafka partitions × Spark workers</h1>",
        "<p class='meta'>Big Data &amp; Predictive Analytics (ST153) · Q2 (RDD) + Q3 (DataFrame/SQL) · "
        "generated %s · driver %s · Spark UI %s</p>"
        % (datetime.now().strftime("%Y-%m-%d %H:%M"), socket.gethostname(),
           runs_ok[0].get("driver_ui", "-") if runs_ok else "-"),
        "<h2>Executive summary</h2><ul>" + "".join("<li>%s</li>" % x for x in bullets) + "</ul>",
        "<h2>Environment</h2>" + df_html(workers),
        "<p>Source: <b>%s</b>, %s messages over %d partitions (per partition %s). "
        "Benchmark corpus: <b>%s</b> messages, identical payloads in every topic.</p>"
        % (SOURCE, format(sum(src_counts.values()), ","), len(src_counts), src_counts,
           format(CORPUS_N, ",")),
    ]
    if warm:
        parts.append("<p>Cold start of a fresh Spark application: <b>%.1f s</b> until %d executor(s) registered and "
                     "ran a first task. Each application pays this once (connector-jar distribution to the workers); "
                     "it is excluded from the measured stages but matters for short jobs.</p>"
                     % (warm["cold_start"]["cold_start_s"], warm["cold_start"]["executors_active"]))
    parts.append("<p><b>Design.</b> P ∈ %s partitions, W ∈ %s executors (2 cores each, <code>spark.cores.max=2W</code>); "
                 "%s messages; explicit schema; <code>spark.sql.shuffle.partitions=16</code>; AQE on. "
                 "speedup = T(W=1,P)/T(W,P); efficiency = speedup / active executors; Jain index over per-executor tasks.</p>"
                 % (P_LIST, W_LIST, format(CORPUS_N, ",")))

    parts.append("<h2>P × W results (balanced topics)</h2>" +
                 df_html(b[["P", "W", "hosts", "makespan_s", "throughput_msg_s", "active_exec", "cold_start_s",
                            "jain_exec", "core_util", "driver_cpu_pct", "exec_mem_mb", "gc_s",
                            "shuffle_wr_mb", "shuffle_rd_mb"]]))
    b["parallel_limit"] = [min(p, a) for p, a in zip(b.P, b.active_exec)]
    b["eff_vs_limit"] = b.speedup / b.parallel_limit.replace(0, np.nan)
    parts.append("<h2>Speedup &amp; efficiency</h2>" +
                 df_html(b[["P", "W", "makespan_s", "throughput_msg_s", "speedup", "efficiency",
                            "active_exec", "parallel_limit", "eff_vs_limit", "jain_exec", "core_util"]]))
    parts.append("<p><b>Reading the table.</b> The ingest stage can only use as many readers as there are "
                 "Kafka partitions, so a partition count below the number of executors caps the achievable "
                 "speedup (<code>parallel_limit = min(P, active executors)</code>). Efficiency is reported both "
                 "against the requested W and against that limit; the second number isolates scheduling and "
                 "shuffle overhead from the hard ingest constraint.</p>")

    if not stages.empty:
        parts.append("<h2>Stage latency</h2>" +
                     df_html(stages[stages.variant == "balanced"]
                             .groupby(["P", "W", "stage"])[["seconds", "tasks", "p50_ms", "p95_ms", "shuffle_wr_mb"]]
                             .mean().reset_index()))

    if len(lats):
        parts.append("<h2>Kafka microbenchmark</h2>" +
                     df_html(lats[["P", "variant", "n", "p50_ms", "p95_ms", "p99_ms", "mean_ms"]]))
    if len(thr):
        parts.append(df_html(thr[["variant", "n", "mean_ms", "msg_per_s", "mb_per_s"]]))
    if len(fan):
        parts.append(df_html(fan[["consumers", "messages", "seconds", "msg_per_s", "per_consumer"]]))

    parts.append("<h2>Weight distribution</h2>")
    wt = pd.DataFrame([{"topic": t, "partitions": len(c), "min": min(c), "max": max(c),
                        "max/min": round(max(c) / max(min(c), 1), 2), "Jain": jain(c)}
                       for t, c in TOPIC_COUNTS.items()])
    parts.append(df_html(wt))
    ex_rows = []
    for r in runs_ok:
        tot = sum(e["tasks"] for e in r["executors"]) or 1
        for e in r["executors"]:
            ex_rows.append({"run": "%s-P%d-W%d" % (r.get("variant", "balanced"), r["P"], r["W"]),
                            "executor": e["id"], "host": e["host"], "tasks": e["tasks"],
                            "share_%": round(100.0 * e["tasks"] / tot, 1),
                            "cpu_s": round(e["duration_ms"] / 1000, 1), "mem_MB": e["memory_used_mb"]})
    if ex_rows:
        parts.append(df_html(pd.DataFrame(ex_rows)))

    parts.append("<h2>Findings</h2><ol>")
    for P in sorted(b.P.unique()):
        s = b[b.P == P].sort_values("W")
        parts.append("<li>P=%d: makespan %s s; throughput %s msg/s; Jain %s; core utilisation %s.</li>" % (
            P, " → ".join("%.1f" % x for x in s.makespan_s),
            " → ".join(format(int(x), ",") for x in s.throughput_msg_s),
            " → ".join("%.2f" % x for x in s.jain_exec),
            " → ".join("%.2f" % (x or 0) for x in s.core_util)))
    if skew_run is not None:
        parts.append("<li>The wiki-keyed topic (P=6) preserves the Q1a skew: partition Jain %.2f, throughput %s msg/s "
                     "at W=2, versus a balanced topic of the same size.</li>"
                     % (skew_run.get("jain_partition", 0), format(int(skew_run["throughput_msg_s"]), ",")))
    parts.append("</ol>")

    parts.append("<h2>Limitations</h2><ul>"
                 "<li>The corpus is a cycled replay of a 109,868-event capture: the composition of wikis/bots is preserved, "
                 "the absolute edit rate of the live stream is not.</li>"
                 "<li>W is the requested executor count (<code>spark.cores.max</code>); the number that actually registered and "
                 "ran tasks is reported per run — the cluster lost one worker mid-experiment (<i>Unable to create executor "
                 "due to null</i> on 10.8.0.5).</li>"
                 "<li>Driver-side psutil covers the broker/driver host; executor CPU, GC and memory come from the Spark REST API.</li>"
                 "<li>The Kafka microbenchmark is client-side produce→consume on the broker host, not an instrumented broker histogram.</li>"
                 "<li>Cold start (connector-jar distribution to executors) is measured separately and excluded from stage times.</li>"
                 "</ul>")
    parts.append("<h2>Reproducibility</h2><p>Notebook <code>wiki_benchmark_q2_q3.ipynb</code> on the Jupyter server; "
                 "raw results <code>%s</code>; microbenchmark <code>%s</code>; plots <code>%s</code>; topics "
                 "<code>%s.*</code> with <code>retention.ms=3600000</code>.</p>"
                 % (RESULT_JSON, MICRO_JSON, PLOT_DIR, PREFIX))

    css = ("body{font-family:Inter,Arial,sans-serif;max-width:1120px;margin:24px auto;padding:0 16px;color:#1a202c;line-height:1.45}"
           "h1{color:#1a365d;margin-bottom:4px}h2{color:#2b6cb0;border-bottom:2px solid #ed8936;padding-bottom:4px;margin-top:30px}"
           ".meta{color:#4a5568;font-size:13px}.t{border-collapse:collapse;font-size:12px;margin:8px 0;width:100%}"
           ".t th{background:#2b6cb0;color:#fff;padding:4px 7px;text-align:right}"
           ".t td{padding:3px 7px;border-bottom:1px solid #e2e8f0;text-align:right}"
           ".t td:first-child,.t th:first-child{text-align:left}"
           "img{max-width:100%;margin:10px 0;border:1px solid #e2e8f0}"
           "code{background:#f7fafc;padding:1px 4px;border-radius:3px}li{margin:3px 0}")

    body = ["<h2>Charts</h2>"]
    for name, p in CH.items():
        body.append("<h3>%s</h3><img src='data:image/png;base64,%s'>" % (name.replace("_", " "), img64(p)))
    doc = ("<!doctype html><html><head><meta charset='utf-8'><title>Wikipedia pipeline benchmark</title>"
           "<style>%s</style></head><body>%s%s</body></html>" % (css, "".join(parts), "".join(body)))
    REPORT_HTML.write_text(doc, encoding="utf-8")
    return REPORT_HTML


report = build_report()
print("report:", report, "|", round(report.stat().st_size / 1024), "KiB")

# also publish a copy next to the notebook so it can be downloaded from the Jupyter server
import shutil
published = Path.cwd() / report.name
shutil.copy(report, published)
print("published copy:", published)
print("runs:", len([r for r in R if not r.get("warmup")]))

report: /tmp/wiki-bench/wiki_q2_q3_benchmark.html | 735 KiB
published copy: /opt/notebooks/wiki_q2_q3_benchmark.html
runs: 13


## 9. Close out

`spark.stop()` is intentionally **not** called here, so the next cell can rebuild the report or re-run a
single combination. Run the last cell when the session is finished to release the cluster for other groups.

In [11]:
# ============================== 11. Optional cleanup ==============================
# spark.stop()            # release executors when finished
# DELETE_TOPICS = False   # set True to remove the benchmark topics
# if DELETE_TOPICS:
#     for t in list(TOPICS.values()) + [SKEW_TOPIC]:
#         with contextlib.suppress(Exception):
#             for _, f in admin.delete_topics([t], operation_timeout=10).items():
#                 f.result()
#     print("topics deleted")
print("done — report at", REPORT_HTML)

done — report at /tmp/wiki-bench/wiki_q2_q3_benchmark.html
